<a href="https://colab.research.google.com/github/Valdan-D/Valdan-D/blob/main/notebooks/cleaning_Danilo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REMORSE — Data Cleaning
**Reptilian Evaluation of Mesozoic Origins: Retrospective Study on Extinction**

Questo notebook esegue la pulizia e la preparazione dei dataset grezzi.
L'output è un CSV processato per dataset, pronto per il caricamento nello star schema SQLite.

Input:  `data/raw/dinos.csv`, `data/raw/plants.csv`  
Output: `data/processed/dinos_clean.csv`, `data/processed/plants_clean.csv`

## 0. Setup

In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerie caricate.')

Librerie caricate.


## 1. Caricamento dei dataset grezzi

In [6]:
dinos_raw = pd.read_csv('data/raw/dinos.csv', low_memory=False)
plants_raw = pd.read_csv('data/raw/plants.csv', low_memory=False)

print(f'dinos_raw:  {dinos_raw.shape[0]:,} righe, {dinos_raw.shape[1]} colonne')
print(f'plants_raw: {plants_raw.shape[0]:,} righe, {plants_raw.shape[1]} colonne')

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/dinos.csv'

## 2. Selezione delle colonne utili

In [ ]:
COLONNE_UTILI = [
    'occurrence_no',
    'collection_no',
    'accepted_name',
    'accepted_rank',
    'early_interval',
    'late_interval',
    'max_ma',
    'min_ma',
    'lat',
    'lng',
    'cc',
    'state',
    'formation',
    'geological_group',
    'phylum',
    'class',
    'order',
    'family',
    'genus',
]

dinos = dinos_raw[COLONNE_UTILI].copy()
plants = plants_raw[COLONNE_UTILI].copy()

print(f'Colonne selezionate: {len(COLONNE_UTILI)}')
print(f'dinos:  {dinos.shape}')
print(f'plants: {plants.shape}')

## 3. Filtro temporale — solo Mesozoico

I dinos includono record fuori dal Mesozoico (Olocene, Pleistocene).
Teniamo solo i record con `max_ma` compreso tra 66 e 252 Ma (Triassico → fine Cretaceo).

In [ ]:
MESOZOICO_MIN = 66.0   # fine Cretaceo (K-Pg)
MESOZOICO_MAX = 252.0  # inizio Triassico

dinos_pre = len(dinos)
dinos = dinos[(dinos['max_ma'] >= MESOZOICO_MIN) & (dinos['max_ma'] <= MESOZOICO_MAX)]

print(f'Dinos prima del filtro:  {dinos_pre:,}')
print(f'Dinos dopo il filtro:    {len(dinos):,}')
print(f'Record rimossi:          {dinos_pre - len(dinos):,}')

## 4. Filtro Aves

Aves rappresenta il 40.2% dei record dinos. Il progetto si concentra sui dinosauri non-aviani del Mesozoico.

In [ ]:
dinos_pre = len(dinos)
dinos = dinos[dinos['class'] != 'Aves']

print(f'Dinos prima del filtro Aves: {dinos_pre:,}')
print(f'Dinos dopo il filtro Aves:   {len(dinos):,}')
print(f'Record Aves rimossi:         {dinos_pre - len(dinos):,}')

## 5. Gestione dei valori nulli

In [ ]:
# Rimuovi righe con cc nullo (pochi record, dato critico)
dinos = dinos.dropna(subset=['cc'])
plants = plants.dropna(subset=['cc'])

# Sostituisci null con 'Unknown' nelle colonne tassonomiche e geografiche
FILL_UNKNOWN = ['state', 'formation', 'geological_group', 'phylum', 'order', 'family', 'genus', 'late_interval']

for col in FILL_UNKNOWN:
    dinos[col] = dinos[col].fillna('Unknown')
    plants[col] = plants[col].fillna('Unknown')

print('Null residui dinos:')
print(dinos.isnull().sum()[dinos.isnull().sum() > 0])
print('\nNull residui plants:')
print(plants.isnull().sum()[plants.isnull().sum() > 0])

## 6. Colonne calcolate

In [ ]:
def assign_period_group(ma):
    if ma >= 201.3:
        return 'Triassic'
    elif ma >= 145.0:
        return 'Jurassic'
    else:
        return 'Cretaceous'

for df in [dinos, plants]:
    df['mid_ma'] = (df['max_ma'] + df['min_ma']) / 2
    df['period_group'] = df['max_ma'].apply(assign_period_group)
    df['has_valid_coords'] = df['lat'].notna() & df['lng'].notna()

dinos['dataset_type'] = 'Dinosauria'
plants['dataset_type'] = 'Plantae'

print('Colonne calcolate aggiunte.')
print(f'\nDistribuzione period_group dinos:')
print(dinos['period_group'].value_counts())
print(f'\nDistribuzione period_group plants:')
print(plants['period_group'].value_counts())

## 7. Riepilogo finale

In [ ]:
print('=== RIEPILOGO CLEANING ===')
print(f'\ndinos_raw  → {dinos_raw.shape[0]:,} righe, {dinos_raw.shape[1]} colonne')
print(f'dinos_clean → {len(dinos):,} righe, {len(dinos.columns)} colonne')
print(f'\nplants_raw  → {plants_raw.shape[0]:,} righe, {plants_raw.shape[1]} colonne')
print(f'plants_clean → {len(plants):,} righe, {len(plants.columns)} colonne')
print(f'\nColonne finali: {list(dinos.columns)}')

## 8. Salvataggio dei dataset processati

In [ ]:
dinos.to_csv('data/processed/dinos_clean.csv', index=False)
plants.to_csv('data/processed/plants_clean.csv', index=False)

print('Salvato: data/processed/dinos_clean.csv')
print('Salvato: data/processed/plants_clean.csv')